In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm, glob
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

import optax
from flax import nnx
import orbax.checkpoint as ocp

import diffrax
from diffrax import diffeqsolve, ODETerm, Dopri5, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize
from jax.experimental.ode import odeint

import jaxpm
from jaxpm.painting import cic_paint, cic_read, compensate_cic
from jaxpm.pm import linear_field, lpt, make_ode_fn, pm_forces, make_ode_fn_diffrax, make_ode_fn
from jaxpm.kernels import fftk, gradient_kernel, invlaplace_kernel, longrange_kernel, invnabla_kernel
from jaxpm.utils import power_spectrum, cross_correlation_coefficients

from jaxpm.nn import MLP
from jaxpm import camels, plotting, hpm, nn

jax.devices("gpu")

[cuda(id=0)]

In [3]:
%load_ext tensorboard
from flax.metrics import tensorboard

2025-01-30 18:43:09.095122: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


# configuration

In [4]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

# CAMELS

In [5]:
SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0"
# SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_1"
# SIM = "/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_2"

out_dict = camels.load_CV_snapshots(
    SIM,
    mesh_per_dim,
    parts_per_dim,
    # i_snapshots=[-2,-1],
    # i_snapshots=range(1, 33+4, 8),
    i_snapshots=range(1, 33+4, 4),
    return_hydro=True,
)

cosmo = out_dict["cosmo"]
scales = out_dict["scales"]

dm_poss = out_dict["dm_poss"]
dm_vels = out_dict["dm_vels"]

gas_poss = out_dict["gas_poss"]
gas_vels = out_dict["gas_vels"]

Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_034.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_050.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_066.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_082.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/IllustrisTNG/CV/CV_0/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


loading snapshots: 100%|██████████| 9/9 [01:36<00:00, 10.78s/it]


# fine tune the HPM-table network with particle positions

In [6]:
model = MLP(
    d_in=3, 
    d_out=1, 
    d_hidden=64, 
    n_hidden=4, 
    rngs=nnx.Rngs(0)
)

In [7]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_mlp_v2.jx")
# checkpointer = ocp.StandardCheckpointer()
# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

In [8]:
summary_writer = tensorboard.SummaryWriter('logs/particle_level')

# per-particle reference
ref_pos = jnp.stack(gas_poss, axis=0)
ref_vel = jnp.stack(gas_vels, axis=0)

# power spectrum reference
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0))
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
ref_rho = vcic_paint(jnp.zeros(mesh_shape), ref_pos)
_, ref_cls = vpower_spectrum(ref_rho)

@nnx.jit
def train_step(model, optimizer):

    def loss_fn(model):
        # res = odeint(
        #     hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, force_type="table"),
        #     [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]],
        #     scales,
        #     rtol=1e-2, 
        #     atol=1e-2
        # )
        res = diffeqsolve(
            terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", force_type="table")),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.01,
            y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
            saveat=SaveAt(ts=scales),
            # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
        res = res.ys
        res = jnp.transpose(res, (1,0,2,3))
    
        # pos_loss = jnp.sum((res[2] - ref_pos)**2, axis=-1)
        pos_loss = jnp.sum((res[2]%64 - ref_pos)**2, axis=-1)

        pos_loss = jnp.where(pos_loss < mesh_per_dim//2, pos_loss, 0.)
        # pos_loss *= jnp.expand_dims(scales, axis=1)
        pos_loss = jnp.mean(pos_loss)

        vel_loss = jnp.sum((res[3] - ref_vel)**2, axis=-1)
        vel_loss = jnp.where(vel_loss < mesh_per_dim//2, vel_loss, 0.)
        # vel_loss *= jnp.expand_dims(scales, axis=1)
        vel_loss = jnp.mean(vel_loss)

        res_rho = vcic_paint(jnp.zeros(mesh_shape), res[2])
        _, res_cls = vpower_spectrum(res_rho)
        cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
        
        # return pos_loss + 0.01 * vel_loss + 0.1 * cl_loss
        # return pos_loss + 0.01 * vel_loss + 0.01 * cl_loss
        return cl_loss
        # return pos_loss + 0.01 * vel_loss
        # return pos_loss + 0.01 * vel_loss
        # return pos_loss + vel_loss
        # return pos_loss

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

losses = []

In [9]:
# %tensorboard --logdir=logs/particle_level

In [10]:
total_steps = 100
# learning_rate = 1e-3
learning_rate = optax.cosine_decay_schedule(
    init_value=1e-3, 
    decay_steps=total_steps, 
    alpha=0.1
)
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

In [11]:
for i in (pbar := tqdm.tqdm(range(total_steps))):    
    loss = train_step(model, optimizer)
    losses.append(loss)

    pbar.set_description(f"Loss: {loss:.4f}")

    # if i % 10 ==0:
    #     summary_writer.scalar('train_loss', loss, i)
    #     summary_writer.scalar('learning_rate', lr_scheduler(i), i)

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

  0%|          | 0/100 [00:00<?, ?it/s]


TypeError: get_hpm_network_ode_fn() got an unexpected keyword argument 'force_type'

In [ ]:
# # see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
# checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_sim_cv0.jx")
# checkpointer = ocp.StandardCheckpointer()

In [ ]:
# _, params = nnx.split(model)
# checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# fine tune the HPM-table network with the field

In [ ]:
model = MLP(
    d_in=3, 
    d_out=1, 
    d_hidden=64, 
    n_hidden=4, 
    rngs=nnx.Rngs(0)
)

In [ ]:
# model = MLP(
#     d_in=3, 
#     d_out=1, 
#     d_hidden=128, 
#     n_hidden=4, 
#     rngs=nnx.Rngs(0)
# )

In [ ]:
summary_writer = tensorboard.SummaryWriter('logs/particle_level')

# field-level reference
vcic_paint = jax.vmap(cic_paint, in_axes=(None,0,None))
vcic_read = jax.vmap(cic_read, in_axes=(0,0))

ref_rho = vcic_paint(jnp.zeros(mesh_shape), gas_poss, cosmo.Omega_b / cosmo.Omega_c)

# power spectrum reference
vpower_spectrum = jax.vmap(
    lambda fields: 
        power_spectrum(
            compensate_cic(fields),
            boxsize=np.array([25.0] * 3),
            kmin=np.pi / 25.0,
            dk=2 * np.pi / 25.0,
        )
)
_, ref_cls = vpower_spectrum(ref_rho)

@nnx.jit
def train_step(model, optimizer):

    def loss_fn(model):
        # res = odeint(
        #     hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, force_type="table"), 
        #     [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]],
        #     scales,
        #     rtol=1e-2, 
        #     atol=1e-2
        # )
        res = diffeqsolve(
            terms=ODETerm(hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, integrator_type="diffrax", force_type="table")),
            solver=LeapfrogMidpoint(),
            t0=scales[0],
            t1=scales[-1],
            dt0=0.01,
            y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
            saveat=SaveAt(ts=scales),
            # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
            max_steps=100,
            stepsize_controller=ConstantStepSize(),
        )
        res = res.ys
        res = jnp.transpose(res, (1,0,2,3))
        
        rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)
        rho_loss = jnp.nanmean((rho - ref_rho)**2)
        
        # eps = 1e-5
        # rho_loss = jnp.mean((jnp.log1p(rho + eps) - jnp.log1p(ref_rho + eps))**2)
        
        # _, res_cls = vpower_spectrum(res_rho)
        # cl_loss = jnp.mean(jnp.sum((res_cls/ref_cls - 1)**2, axis=-1))
        
        return rho_loss
        # return rho_loss + 0.1 * cl_loss

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)

    return loss

losses = []

In [ ]:
# def loss_fn(model):
#     res = odeint(
#         hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, force_type="table"), 
#         [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]],
#         scales,
#         rtol=1e-2, 
#         atol=1e-2
#     )
    
#     rho = vcic_paint(jnp.zeros(mesh_shape), res[2], cosmo.Omega_b / cosmo.Omega_c)

#     # diff = rho - ref_rho
#     # print(rho)
#     # print(ref_rho)
#     # print(diff[-1])
    
#     rho_loss = jnp.nanmean((rho - ref_rho)**2)

#     # print(rho - ref_rho)
    
#     # eps = 1e-5
#     # rho_loss = jnp.mean((jnp.log1p(rho + eps) - jnp.log1p(ref_rho + eps))**2)
        
#     return rho_loss

# loss_fn(model)

In [ ]:
total_steps = 100
# learning_rate = 1e-3
# learning_rate = 1e-4
learning_rate = optax.cosine_decay_schedule(
    init_value=1e-3, 
    decay_steps=total_steps, 
    alpha=0.0
)
clip_norm = 1

optimizer = nnx.Optimizer(
    model,
    optax.chain(
        optax.clip_by_global_norm(clip_norm),
        optax.adam(learning_rate)
    )
)

In [ ]:
for i in (pbar := tqdm.tqdm(range(total_steps))):    
    loss = train_step(model, optimizer)
    losses.append(loss)

    pbar.set_description(f"Loss: {loss:.4f}")

    # if i % 10 ==0:
    #     summary_writer.scalar('train_loss', loss, i)
    #     summary_writer.scalar('learning_rate', lr_scheduler(i), i)

fig, ax = plt.subplots()
ax.plot(losses)
ax.set(yscale="log")

In [ ]:
# see https://flax.readthedocs.io/en/latest/guides/checkpointing.html
checkpoint_file = os.path.join(os.getcwd(), "checkpoints/hpm_sim_mlp.jx")
checkpointer = ocp.StandardCheckpointer()

In [ ]:
_, params = nnx.split(model)
checkpointer.save(checkpoint_file, params, force=True)

In [ ]:
# abstract_model = nnx.eval_shape(lambda: model)
# graphdef, abstract_params = nnx.split(abstract_model)

# params = checkpointer.restore(checkpoint_file, abstract_params)
# model = nnx.merge(graphdef, params)

# run the simulation

In [ ]:
pm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "odeint", gravity_only=True)

res = odeint(pm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, rtol=1e-5, atol=1e-5)
pm_dm_poss, pm_dm_vels, pm_gas_poss, pm_gas_vels = res[0], res[1], res[2], res[3]

In [ ]:
hpm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "odeint")

res = odeint(hpm_ode, [dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], scales, rtol=1e-5, atol=1e-5, mxstep=20)
hpm_dm_poss, hpm_dm_vels, hpm_gas_poss, hpm_gas_vels = res[0], res[1], res[2], res[3]

In [ ]:
plotting.compare_particle_evolution(
    mesh_shape, 
    scales, 
    jnp.stack([gas_poss, pm_gas_poss, hpm_gas_poss], axis=0), 
    title="gas",
    col_titles=["CAMELS", "gravity", "gravity + pressure"],
    include_pk=True,
    include_reference=True,
)

In [ ]:
# k, cross_i = cross_correlation_coefficients(
#       (cic_paint(jnp.zeros(mesh_shape), gas_poss[-1])),
#       (cic_paint(jnp.zeros(mesh_shape), pm_gas_poss[-1])),
#       boxsize=np.array([25.] * 3),
#       kmin=np.pi / 25.,
#       dk=2 * np.pi / 25.
# )

In [ ]:
# plotting.compare_particle_evolution(
#     mesh_shape, 
#     scales, 
#     jnp.stack([gas_poss, hpm_gas_poss], axis=0), 
#     title="gas",
#     col_titles=["CAMELS", "gravity + pressure"],
#     include_pk=True,
# )

In [ ]:
# plotting.compare_particle_evolution(
#     mesh_shape, 
#     scales, 
#     jnp.stack([gas_poss, pm_gas_poss], axis=0), 
#     title="gas",
#     col_titles=["CAMELS", "gravity"],
#     include_pk=True,
# )

### diffrax

In [ ]:
# pm_ode = hpm.get_hpm_network_ode_fn(model, mesh_shape, cosmo, "diffrax", gravity_only=True)

# res = diffeqsolve(
#         terms=ODETerm(pm_ode),
#         solver=LeapfrogMidpoint(),
#         # solver=Dopri5(),
#         t0=scales[0],
#         t1=scales[-1],
#         dt0=0.01,
#         y0=jnp.stack([dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]], axis=0),
#         saveat=SaveAt(ts=scales),
#         # adjoint=diffrax.RecursiveCheckpointAdjoint(checkpoints),
#         max_steps=100,
#         stepsize_controller=ConstantStepSize(),
# )

# pm_dm_poss, pm_dm_vels, pm_gas_poss, pm_gas_vels = res.ys[:,0], res.ys[:,1], res.ys[:,2], res.ys[:,3]